In [ ]:
#Instalação
!pip install yfinance==0.2.41
!pip install crewai==0.28.8
!pip install langchain==0.1.20
!pip install langchain_openai==0.1.7
!pip install langchain_community==0.0.38
!pip install duckduckgo-search==5.3.0


  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached embedchain-0.1.128-py3-none-any.whl.metadata (9.2 kB)
  Using cached instructor-0.5.2-py3-none-any.whl.metadata (10 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_http-1.40.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached regex-2023.12.25-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached chromadb-0.5.23-py3-none-any.whl.metadata (6.8 kB)
  Using cached gptcache-0.1.44-py3-none-any.whl.metadata (24 kB)
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
  Using cached embedchain-0.1.127-py3-none-any.whl.metadata (9.3 kB)
  Using cached cohere-5.21.1-py3-none-any.whl.metadata (3.6 kB)


  error: subprocess-exited-with-error
  
  × Building wheel for chroma-hnswlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [5 lines of output]
      running bdist_wheel
      running build
      running build_ext
      building 'hnswlib' extension
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for chroma-hnswlib
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> chroma-hnswlib


In [2]:
#Import das Libs
import json
import os
from datetime import datetime

import yfinance as yf

from crewai import Agent, Task, Crew, Process

from langchain.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults


from IPython.display import display, Markdown

ModuleNotFoundError: No module named 'crewai'

In [ ]:
def fetch_stock_price(ticket):
    stock = yf.download(ticket, start="2023-08-08", end="2024-08-08")
    return stock

yahoo_finance_tool = Tool(
    name = "Yahoo Finance Tool",
    description = "Fetches stocks prices for {ticket} from the last year about a specific stock from Yahoo Finance API",
    func = lambda ticket: fetch_stock_price(ticket)
)

In [ ]:
# IMPORTANDO OPENAI LLM - GPT

os.environ['OPENAI_API_KEY'] = ""
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
stockPriceAnalysisAgent = Agent(
    role= "Senior stock price Analyst",
    goal="Find the {ticket} stock price and analyses treends",
    backstory="" \
    "You are a senior stock price analyst with extensive experience in analyzing the price of an specific stock and make predictions about its futere price",
    verbose=True,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[yahoo_finance_tool],
    allow_delegation=False
)

In [ ]:
getStockPriceTask = Task(
    description="Analyze the stock {ticket} price and create a trend analyses of up, down or sideways",
    expected_output="Specify the current trend stock price - up, down or sideways. eg. stock= 'AAPLM, price UP'",
    agent=stockPriceAnalysisAgent
)

In [ ]:
##IMPORTANDO A TOLL DE SEARCH - DUCKDUCKGO 
search_tool =  DuckDuckGoSearchResults(backend='news', num_results=10)

In [ ]:
newsAnalystAgent = Agent(
    role="Stock News Analyst",
    goal="Create a summary of the market news related to the stock {ticket} company. Specify the current trend - up, down or sideways with the news context. For each request stock asset, specify a numbet between 0 and 100, where 0 is extreme fear and 100 is extrene greed.",
    backstory="You are highly experiencied in analyzing the market trends and news and have tracked assest for more then 10 years. You are also master level analyts in the tradicioonal markets and have deep understanding of human psychology. You undestand news, theirs tittles and information, but you look at those with a health dose of skepticism. You consider also the source of the news articles",
    verbose=True,
    llm=llm,
    max_iter=10,
    memory=True,
    tools=[search_tool],
    allow_delegation=False

)

In [ ]:
get_newsTask = Task(
    description="Take the stock and always include BTC to it (if not request). Use the search tool to search each one individually. The current date is {datetime.now()}. Compose the results into a helpfull report",
    expected_output="A summary of the overall market and one sentence summary for each request asset. Include a fear/greed score for each asset based on the news. Use format: <STOCK ASSET> <SUMARRY BASED ON THE NEWS> <TREND PREDICTION> <FEAR/GREED SCORE>",
    agent=newsAnalystAgent
)

In [ ]:
stockAnalystWriterAgent = Agent(
    role="Senior Stock Analyst Writer",
    goal="Analyze the trends price and news and write an insighfull compelling and informative 3 paragraph long newsletter based on the stock report and price trend.",
    backstory="You are widely accepted the best stock analyst in the market. You understand complex conceptsd and create compelling stories and narratives tha resonate with wider audiences. You understand macro factors and combine multiple theories - eg. cycle theory and fumdamental analyses. You are able to hold multiple opinios when analyzing anything",
    verbose=True,
    llm=llm,
    max_iter=5,
    memory=True,
    allow_delegation=True
    )

In [ ]:
writeAnalysisTask = Task(
    description="Use the stock price trend and the stock news report to create an analyses and write the newsletter about the {ticket} company that is brief and highlights the most important points. Focus on the stock price, news and fear/greed score. What are the near future considerations? Include the previous analyses of stock trend and news summary",
    expected_output="An eloquent 3 paragraph newsletter formated as markdown in an easy readable manner. It should contain: -3 bullets executive summary -Introducion: set the overral picture and spike up the interest. - main part provides the meat of the analysis including the news anda fear/greed scores." \
    "- summary: keys facts and concrete future trend prediction - up, down or sideways.",
    agent = stockAnalystWriterAgent,
    context = [getStockPriceTask, get_newsTask]
)

In [ ]:
crew = Crew(
    agents=[stockPriceAnalysisAgent, newsAnalystAgent, stockAnalystWriterAgent],
    tasks=[getStockPriceTask, get_newsTask, writeAnalysisTask],
    verbose=2,
    process= Process.hierarchical,
    full_output=True,
    share_crew=True,
    manager_llm=llm,
    max_iter=15
)

In [ ]:
results = crew.kickoff(inputs={'ticket': 'AAPL'})

In [ ]:
list(results.keys())

In [ ]:
results['final_output']

In [ ]:
len(results['tasks_outputs_'])

In [ ]:
Markdown(results['final_output'])